# 🏛️ LegalComparator — Training & Export ke Hugging Face

Notebook ini melanjutkan dari notebook sebelumnya:
- Melatih model perbandingan hukum
- Mengemas model agar bisa didownload orang lain
- Mempublish ke Hugging Face Hub

**Prasyarat:** Sudah menjalankan notebook preprocessing sebelumnya
dan memiliki file:
- `corpus_structured.json`
- `embeddings_labse.npy`

---

## 📦 CELL 1 — Instalasi & Setup

In [ ]:
# ============================================================
# CELL 1 — INSTALASI
# ============================================================

!pip install -q sentence-transformers
!pip install -q keybert
!pip install -q huggingface_hub
!pip install -q scikit-learn
!pip install -q gradio

print('✅ Semua library berhasil diinstall!')

## 📁 CELL 2 — Upload File dari Notebook Sebelumnya

In [ ]:
# ============================================================
# CELL 2 — UPLOAD HASIL NOTEBOOK SEBELUMNYA
# Upload corpus_structured.json dan embeddings_labse.npy
# ============================================================

from google.colab import files

print('📂 Upload file hasil notebook preprocessing:')
print('   1. corpus_structured.json')
print('   2. embeddings_labse.npy')
print()

uploaded = files.upload()

print(f'\n✅ {len(uploaded)} file berhasil diupload:')
for fname in uploaded:
    print(f'   📄 {fname}')

## 🧠 CELL 3 — Load LegalComparator Model

In [ ]:
# ============================================================
# CELL 3 — INSTALL & LOAD MODEL DARI GITHUB
# ============================================================

# Download source model langsung dari GitHub
!wget -q https://raw.githubusercontent.com/YOUR_USERNAME/indonesian-legal-nlp/main/src/legal_comparator.py

# --- ATAU copy manual jika belum ada di GitHub ---
# Paste isi file legal_comparator.py ke sini jika perlu

import sys
sys.path.insert(0, '.')

from legal_comparator import LegalComparator

# Inisialisasi model
model = LegalComparator(
    embedding_model  = 'sentence-transformers/LaBSE',
    threshold_tinggi = 0.80,
    threshold_sedang = 0.65
)

# Load corpus + embedding yang sudah ada
model.load(
    corpus_path    = 'corpus_structured.json',
    embedding_path = 'embeddings_labse.npy'    # tidak perlu hitung ulang!
)

print('\n📚 Dokumen dalam corpus:')
for doc in model.daftar_dokumen():
    info = model.info_dokumen(doc)
    print(f'   {doc:<20} : {info["jumlah_pasal"]} pasal ({info["bahasa"].upper()})')

## 🔍 CELL 4 — Demo Fitur 1: Similarity Score

In [ ]:
# ============================================================
# CELL 4 — DEMO FITUR 1: SIMILARITY SCORE ANTAR PASAL
# Bandingkan dua pasal spesifik secara cross-lingual
# ============================================================

import json

print('=' * 60)
print('FITUR 1: SIMILARITY SCORE ANTAR PASAL')
print('=' * 60)

# Contoh: UNCAC Article 15 (Bribery) vs UU 31/1999 Pasal 5 (Suap)
hasil = model.similarity_score(
    pasal_a_id = 'UNCAC_article_15',    # Article tentang Bribery
    pasal_b_id = 'UU_31_1999_pasal_5'  # Pasal tentang Suap
)

print(json.dumps(hasil, indent=2, ensure_ascii=False))

print('\n' + '─' * 60)
print('Coba pasangan lain: UNCAC Article 20 (Illicit enrichment)')
print('─' * 60)

# Article 20 = Illicit Enrichment (pengayaan tidak sah)
# Ini adalah salah satu GAP terbesar Indonesia
hasil2 = model.similarity_score(
    pasal_a_id = 'UNCAC_article_20',
    pasal_b_id = 'UU_20_2001_pasal_12'
)
print(json.dumps(hasil2, indent=2, ensure_ascii=False))

## 📊 CELL 5 — Demo Fitur 2: Gap Analysis Lengkap

In [ ]:
# ============================================================
# CELL 5 — DEMO FITUR 2: GAP ANALYSIS
# Bandingkan seluruh dokumen secara otomatis
# ============================================================

print('=' * 60)
print('FITUR 2: GAP ANALYSIS OTOMATIS')
print('=' * 60)

# --- Perbandingan 1: UNCAC vs UU Tipikor ---
print('\n🔹 UNCAC vs UU Tipikor (UU 31/1999 + UU 20/2001)')
hasil_tipikor = model.compare(
    doc_a     = 'UNCAC',
    doc_b     = 'UU_31_1999',
    top_n_gap = 5
)

# --- Perbandingan 2: UNCAC vs UU KPK ---
print('\n🔹 UNCAC vs UU KPK (UU 30/2002)')
hasil_kpk = model.compare(
    doc_a     = 'UNCAC',
    doc_b     = 'UU_30_2002',
    top_n_gap = 5
)

# --- Perbandingan 3: Antar UU Indonesia ---
print('\n🔹 UU Tipikor 1999 vs UU Tipikor 2001 (evolusi hukum)')
hasil_evolusi = model.compare(
    doc_a     = 'UU_31_1999',
    doc_b     = 'UU_20_2001',
    top_n_gap = 5
)

## 🔎 CELL 6 — Demo Fitur 3: Search Pasal by Topik

In [ ]:
# ============================================================
# CELL 6 — DEMO FITUR 3: SEMANTIC SEARCH
# Cari pasal berdasarkan topik/pertanyaan
# ============================================================

print('=' * 60)
print('FITUR 3: SEARCH PASAL BERDASARKAN TOPIK')
print('=' * 60)

# Query Bahasa Indonesia
print('\n🔎 Query: "suap kepada pejabat negara"')
hasil_suap = model.search(
    query   = 'suap kepada pejabat negara',
    top_n   = 5
)

# Query Bahasa Inggris (cross-lingual)
print('\n🔎 Query: "anti-money laundering financial institutions"')
hasil_ml = model.search(
    query   = 'anti-money laundering financial institutions',
    top_n   = 5
)

# Query spesifik dalam satu dokumen
print('\n🔎 Query: "pemulihan aset hasil kejahatan" (di UNCAC saja)')
hasil_aset = model.search(
    query   = 'asset recovery proceeds of crime',
    dokumen = 'UNCAC',
    top_n   = 5
)

# Query bahasa bebas
print('\n🔎 Query: "penggelapan uang negara oleh pejabat"')
hasil_gelap = model.search(
    query   = 'penggelapan uang negara oleh pejabat',
    top_n   = 5
)

## 📝 CELL 7 — Demo Fitur 4: Ringkasan Perbedaan

In [ ]:
# ============================================================
# CELL 7 — DEMO FITUR 4: SUMMARIZE PERBEDAAN
# Tiga gaya output: naratif, poin, tabel
# ============================================================

print('=' * 60)
print('FITUR 4: RINGKASAN PERBEDAAN')
print('=' * 60)

# Gaya naratif (default) — cocok untuk laporan
print('\n📄 GAYA: NARATIF')
print(model.summarize_gap(
    hasil_tipikor,
    top_n_gap = 5,
    gaya      = 'naratif'
))

# Gaya poin (Markdown) — cocok untuk README GitHub
print('\n📄 GAYA: POIN (Markdown)')
print(model.summarize_gap(
    hasil_tipikor,
    top_n_gap = 5,
    gaya      = 'poin'
))

# Gaya tabel — cocok untuk presentasi
print('\n📄 GAYA: TABEL')
print(model.summarize_gap(
    hasil_tipikor,
    top_n_gap = 10,
    gaya      = 'tabel'
))

## 📦 CELL 8 — Kemas Model untuk Distribusi

In [ ]:
# ============================================================
# CELL 8 — KEMAS SEMUA FILE UNTUK DISTRIBUSI
# Membuat package yang siap diupload ke Hugging Face Hub
# ============================================================

import os, json, shutil
from datetime import datetime

# Buat direktori model
MODEL_DIR = 'indonesian-legal-comparator'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(f'{MODEL_DIR}/src', exist_ok=True)

# ── 1. Salin file model utama ──
shutil.copy('legal_comparator.py', f'{MODEL_DIR}/src/legal_comparator.py')

# ── 2. Salin corpus dan embedding ──
shutil.copy('corpus_structured.json', f'{MODEL_DIR}/corpus_structured.json')
shutil.copy('embeddings_labse.npy',   f'{MODEL_DIR}/embeddings_labse.npy')

# ── 3. Buat model config ──
config = {
    'model_name'       : 'indonesian-legal-comparator',
    'versi'            : '1.0.0',
    'dibuat_pada'      : datetime.now().isoformat(),
    'embedding_model'  : 'sentence-transformers/LaBSE',
    'threshold_tinggi' : 0.80,
    'threshold_sedang' : 0.65,
    'bahasa'           : ['id', 'en'],
    'domain'           : 'hukum antikorupsi indonesia',
    'dokumen_dalam_corpus': model.daftar_dokumen(),
    'total_pasal'      : len(model.semua_pasal),
    'fitur'            : [
        'similarity_score',
        'compare (gap_analysis)',
        'search',
        'summarize_gap'
    ],
    'dependensi': [
        'sentence-transformers>=2.2.0',
        'keybert>=0.7.0',
        'scikit-learn>=1.0.0',
        'numpy>=1.21.0'
    ]
}

with open(f'{MODEL_DIR}/config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

# ── 4. Buat requirements.txt ──
req_content = '\n'.join(config['dependensi']) + '\n'
with open(f'{MODEL_DIR}/requirements.txt', 'w') as f:
    f.write(req_content)

# ── 5. Buat __init__.py ──
init_content = '''
from .src.legal_comparator import LegalComparator, HasilPerbandingan, HasilSearch, Pasal
__version__ = "1.0.0"
__all__ = ["LegalComparator", "HasilPerbandingan", "HasilSearch", "Pasal"]
'''
with open(f'{MODEL_DIR}/__init__.py', 'w') as f:
    f.write(init_content)

# ── 6. Buat README model card (Hugging Face format) ──
readme = '''---
language:
  - id
  - en
tags:
  - legal-nlp
  - anti-corruption
  - indonesian-law
  - sentence-similarity
  - cross-lingual
license: mit
---

# 🏛️ Indonesian Legal Comparator

Model NLP untuk membandingkan dokumen hukum antikorupsi Indonesia
dengan standar internasional UNCAC secara otomatis.

## ✨ Fitur

| Fitur | Deskripsi |
|---|---|
| `similarity_score()` | Skor kemiripan semantik antar dua pasal (cross-lingual EN↔ID) |
| `compare()` | Gap analysis lengkap antara dua dokumen hukum |
| `search()` | Cari pasal berdasarkan topik/query bebas |
| `summarize_gap()` | Ringkasan perbedaan dalam format naratif/poin/tabel |

## 📦 Instalasi

```bash
pip install sentence-transformers scikit-learn keybert numpy
```

## 🚀 Cara Pakai

```python
from src.legal_comparator import LegalComparator

# 1. Inisialisasi model
model = LegalComparator()

# 2. Load corpus
model.load(
    corpus_path    = "corpus_structured.json",
    embedding_path = "embeddings_labse.npy"
)

# 3. Similarity score dua pasal spesifik
hasil = model.similarity_score(
    pasal_a_id = "UNCAC_article_15",
    pasal_b_id = "UU_31_1999_pasal_5"
)
print(hasil["score"], hasil["interpretasi"])

# 4. Gap analysis penuh
compare = model.compare(doc_a="UNCAC", doc_b="UU_31_1999")

# 5. Ringkasan perbedaan
print(model.summarize_gap(compare, gaya="naratif"))

# 6. Cari pasal berdasarkan topik
search = model.search("suap kepada pejabat negara", top_n=5)
```

## 📚 Dokumen dalam Corpus

| Label | Dokumen | Tahun | Bahasa |
|---|---|---|---|
| `UNCAC` | UN Convention Against Corruption | 2003 | EN |
| `UU_7_2006` | UU Ratifikasi UNCAC | 2006 | ID |
| `UU_31_1999` | UU Tipikor | 1999 | ID |
| `UU_20_2001` | Amandemen UU Tipikor | 2001 | ID |
| `UU_28_1999` | UU Penyelenggaraan Negara Bersih KKN | 1999 | ID |
| `UU_30_2002` | UU KPK | 2002 | ID |
| `UU_19_2019` | UU Revisi KPK | 2019 | ID |

## 🧠 Model yang Digunakan

- **Embedding**: `sentence-transformers/LaBSE` (109 bahasa, cross-lingual)
- **Keyword Extraction**: `KeyBERT` + `paraphrase-multilingual-MiniLM-L12-v2`
- **Similarity**: Cosine similarity dengan threshold adaptif

## 📄 Lisensi

MIT License — bebas digunakan untuk penelitian dan pengembangan.
'''

with open(f'{MODEL_DIR}/README.md', 'w', encoding='utf-8') as f:
    f.write(readme)

# Tampilkan isi direktori
print(f'📦 Package model berhasil dibuat: {MODEL_DIR}/')
print()
for root, dirs, fls in os.walk(MODEL_DIR):
    level = root.replace(MODEL_DIR, '').count(os.sep)
    indent = '   ' * level
    print(f'{indent}📁 {os.path.basename(root)}/')
    for f in fls:
        fpath = os.path.join(root, f)
        size  = os.path.getsize(fpath) / 1024
        print(f'{indent}   📄 {f} ({size:.1f} KB)')

## 🤗 CELL 9 — Upload ke Hugging Face Hub

In [ ]:
# ============================================================
# CELL 9 — UPLOAD KE HUGGING FACE HUB
# Prasyarat: punya akun Hugging Face
# Token: https://huggingface.co/settings/tokens
# ============================================================

from huggingface_hub import HfApi, login
from google.colab import userdata
import os

# ── Login ke Hugging Face ──
# Cara 1: Pakai Colab Secrets (DISARANKAN — lebih aman)
#   Buka kunci 🔑 di sidebar Colab → New secret
#   Name: HF_TOKEN, Value: token dari huggingface.co/settings/tokens

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('✅ Token ditemukan dari Colab Secrets')
except:
    # Cara 2: Input manual (kurang aman, jangan share notebook)
    HF_TOKEN = input('Masukkan Hugging Face token Anda: ')

login(token=HF_TOKEN)

# ── Konfigurasi Repository ──
HF_USERNAME  = input('\nMasukkan username Hugging Face Anda: ')
REPO_NAME    = 'indonesian-legal-comparator'
REPO_ID      = f'{HF_USERNAME}/{REPO_NAME}'

print(f'\n🚀 Akan mengupload ke: https://huggingface.co/{REPO_ID}')

# ── Buat Repository ──
api = HfApi()

try:
    api.create_repo(
        repo_id = REPO_ID,
        private = False,   # Set True jika ingin private
        exist_ok= True
    )
    print(f'✅ Repository dibuat/sudah ada: {REPO_ID}')
except Exception as e:
    print(f'⚠️  {e}')

# ── Upload semua file ──
print(f'\n📤 Mengupload semua file...')

api.upload_folder(
    folder_path = MODEL_DIR,
    repo_id     = REPO_ID,
    repo_type   = 'model',
    commit_message = 'Upload Indonesian Legal Comparator v1.0.0'
)

print(f'\n🎉 Model berhasil dipublish!')
print(f'   🔗 URL: https://huggingface.co/{REPO_ID}')
print(f'\n📥 Cara orang lain menggunakan model ini:')
print(f'   from huggingface_hub import snapshot_download')
print(f'   path = snapshot_download(repo_id="{REPO_ID}")')
print(f'   from src.legal_comparator import LegalComparator')
print(f'   model = LegalComparator().load(path + "/corpus_structured.json",')
print(f'                                  path + "/embeddings_labse.npy")')

## 🌐 CELL 10 — Demo App Gradio (Antarmuka Web)

In [ ]:
# ============================================================
# CELL 10 — GRADIO DEMO APP
# Antarmuka web interaktif untuk demonstrasi model
# Bisa di-deploy ke Hugging Face Spaces
# ============================================================

import gradio as gr
import json

# ── Helper functions untuk Gradio ──

def fn_similarity(pasal_a_id, pasal_b_id):
    """Wrapper similarity_score untuk Gradio."""
    try:
        hasil = model.similarity_score(pasal_a_id.strip(), pasal_b_id.strip())
        output = (
            f"📊 Similarity Score: {hasil['score']}\n"
            f"📋 Level           : {hasil['level']}\n"
            f"💬 Interpretasi    : {hasil['interpretasi']}\n\n"
            f"─── Preview Pasal A ───\n{hasil['preview_a']}\n\n"
            f"─── Preview Pasal B ───\n{hasil['preview_b']}"
        )
        return output
    except Exception as e:
        return f'❌ Error: {str(e)}'


def fn_gap_analysis(doc_a, doc_b, top_n, gaya):
    """Wrapper compare + summarize untuk Gradio."""
    try:
        hasil = model.compare(doc_a, doc_b, top_n_gap=int(top_n))
        return model.summarize_gap(hasil, top_n_gap=int(top_n), gaya=gaya)
    except Exception as e:
        return f'❌ Error: {str(e)}'


def fn_search(query, dokumen, top_n):
    """Wrapper search untuk Gradio."""
    try:
        dok_filter = dokumen if dokumen != 'Semua Dokumen' else None
        hasil = model.search(query, dokumen=dok_filter, top_n=int(top_n))
        lines = [f'🔎 Hasil pencarian: "{query}"\n']
        for h in hasil.hasil:
            lines.append(
                f"#{h['rank']} [{h['score']:.3f}] {h['dokumen']} — Pasal {h['nomor_pasal']}\n"
                f"{h['preview']}\n"
            )
        return '\n'.join(lines)
    except Exception as e:
        return f'❌ Error: {str(e)}'


# ── Daftar pilihan dokumen ──
DOKUMEN_LIST = model.daftar_dokumen()

# ── Bangun UI Gradio ──
with gr.Blocks(
    title='🏛️ Indonesian Legal Comparator',
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # 🏛️ Indonesian Legal Comparator
    **Model NLP untuk membandingkan hukum antikorupsi Indonesia vs UNCAC**
    Mendukung pencarian dan perbandingan cross-lingual (Bahasa Indonesia & Inggris)
    """)

    with gr.Tabs():

        # Tab 1: Similarity Score
        with gr.Tab('📐 Similarity Score'):
            gr.Markdown('### Bandingkan dua pasal spesifik')
            gr.Markdown('Contoh ID pasal: `UNCAC_article_15`, `UU_31_1999_pasal_5`')
            with gr.Row():
                inp_a = gr.Textbox(label='ID Pasal A', placeholder='UNCAC_article_15')
                inp_b = gr.Textbox(label='ID Pasal B', placeholder='UU_31_1999_pasal_5')
            btn_sim = gr.Button('🔍 Hitung Similarity', variant='primary')
            out_sim = gr.Textbox(label='Hasil', lines=12)
            btn_sim.click(fn_similarity, inputs=[inp_a, inp_b], outputs=out_sim)

        # Tab 2: Gap Analysis
        with gr.Tab('📊 Gap Analysis'):
            gr.Markdown('### Bandingkan dua dokumen hukum secara menyeluruh')
            with gr.Row():
                sel_a  = gr.Dropdown(DOKUMEN_LIST, label='Dokumen A', value='UNCAC')
                sel_b  = gr.Dropdown(DOKUMEN_LIST, label='Dokumen B', value='UU_31_1999')
            with gr.Row():
                n_gap  = gr.Slider(3, 20, value=5, step=1, label='Jumlah Gap Ditampilkan')
                gaya   = gr.Radio(
                    ['naratif', 'poin', 'tabel'],
                    value='naratif',
                    label='Gaya Ringkasan'
                )
            btn_gap = gr.Button('📊 Analisis Gap', variant='primary')
            out_gap = gr.Textbox(label='Hasil Gap Analysis', lines=25)
            btn_gap.click(fn_gap_analysis, inputs=[sel_a, sel_b, n_gap, gaya], outputs=out_gap)

        # Tab 3: Search
        with gr.Tab('🔎 Search Pasal'):
            gr.Markdown('### Cari pasal berdasarkan topik (bahasa bebas)')
            with gr.Row():
                inp_q   = gr.Textbox(
                    label='Query / Topik',
                    placeholder='contoh: suap pejabat negara / bribery of public official'
                )
                sel_dok = gr.Dropdown(
                    ['Semua Dokumen'] + DOKUMEN_LIST,
                    label='Filter Dokumen',
                    value='Semua Dokumen'
                )
            n_res   = gr.Slider(1, 10, value=5, step=1, label='Jumlah Hasil')
            btn_src = gr.Button('🔎 Cari Pasal', variant='primary')
            out_src = gr.Textbox(label='Hasil Pencarian', lines=20)
            btn_src.click(fn_search, inputs=[inp_q, sel_dok, n_res], outputs=out_src)

    gr.Markdown("""
    ---
    📚 **Dokumen dalam corpus:** UNCAC · UU 7/2006 · UU 31/1999 · UU 20/2001 · UU 28/1999 · UU 30/2002 · UU 19/2019
    🧠 **Model:** LaBSE (cross-lingual EN↔ID) · KeyBERT · Cosine Similarity
    """)

# Jalankan demo
print('🚀 Menjalankan Gradio demo...')
demo.launch(
    share=True,          # Buat public URL
    debug=False,
    show_error=True
)

print('\n💡 Untuk deploy ke Hugging Face Spaces:')
print('   1. Buat Space baru di huggingface.co/spaces')
print('   2. Upload app.py (isi Cell 10 ini) + requirements.txt')
print('   3. Space otomatis running secara gratis!')